# Empire Play — Importador

Roda 100% no navegador via Google Colab. Sem chave de conta de servico.

**Como usar:**
1. Execute celula por celula (Shift+Enter)
2. Na celula 2, clique no link e autorize com sua conta Google
3. Cole a SUPABASE_KEY quando solicitado na celula 3

In [ ]:
# Celula 1 — Instalar dependencias
!pip install -q gspread supabase

In [ ]:
# Celula 2 — Autenticar com Google
from google.colab import auth
auth.authenticate_user()
print('OK Autenticado com Google')

In [ ]:
# Celula 3 — Configurar Supabase
from getpass import getpass
SUPABASE_URL = 'https://rqwprnvlrobabfotmmnf.supabase.co'
SUPABASE_KEY = getpass('Cole sua Supabase service_role key: ')
SHEET_ID     = '1XYa6Pzd-lou3fzqaZgjhBYNb3Je2PB9Slu7ozzOghUo'
print('OK Configurado')

In [ ]:
# Celula 4 — Conectar ao Sheets e Supabase
import gspread
from google.auth import default
from supabase import create_client
creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/spreadsheets.readonly',
    'https://www.googleapis.com/auth/drive.readonly'])
gc = gspread.authorize(creds)
sh = gc.open_by_key(SHEET_ID)
sb = create_client(SUPABASE_URL, SUPABASE_KEY)
print('OK Conectado ao Sheets e Supabase')

In [ ]:
# Celula 5 — Funcoes auxiliares
def ler_aba(*nomes):
    for nome in nomes:
        try:
            ws = sh.worksheet(nome)
            rows = ws.get_all_records(default_blank=None)
            print(f'  OK Aba "{nome}" -- {len(rows)} linhas')
            return rows
        except gspread.WorksheetNotFound:
            continue
    print(f'  AVISO Aba nao encontrada: {nomes}')
    return []

def norm(schema, obj):
    return {k: (obj.get(k) or None) for k in schema}

def upsert(tabela, registros):
    if not registros:
        print(f'  AVISO Sem registros para {tabela}'); return
    for i in range(0, len(registros), 500):
        lote = registros[i:i+500]
        try:
            sb.table(tabela).upsert(lote).execute()
            print(f'  OK {tabela} -- lote {i//500+1} ({len(lote)} linhas)')
        except Exception as e:
            print(f'  ERRO {tabela} lote {i//500+1}: {e}')

def n(val):
    try: return int(val) if val else None
    except: return None

print('OK Funcoes carregadas')

In [ ]:
# Celula 6 — MUSICAS
print('--- Musicas')
S = {'Nome':None,'Artista':None,'Album':None,'Capa da Musica':None,
     'Link do audio':None,'telegram_file_id':None,'telegram_topic_id':None,
     'genero':None,'tipo_single':None,'tipo_musica':None,'data_lancamento':None,'ordem':None}
rows = ler_aba('Musicas')
reg = []
for r in rows:
    nome = r.get('Nome da musica') or r.get('Nome')
    if not nome: continue
    reg.append(norm(S, {
        'Nome': nome,
        'Artista': r.get('ACT PRINCIPAL') or r.get('Artista'),
        'Album': r.get('ALBUM'),
        'Capa da Musica': r.get('Capa da musica'),
        'Link do audio': r.get('Link do audio'),
        'telegram_file_id': r.get('ID do arquivo'),
        'telegram_topic_id': n(r.get('ID do topico')),
        'genero': r.get('GENERO'),
        'tipo_single': r.get('TIPO DE SINGLE'),
        'tipo_musica': r.get('TIPO DE MUSICA'),
        'data_lancamento': str(r.get('Data de lancamento', '') or '') or None,
        'ordem': n(r.get('Ordem')),
    }))
upsert('Musicas', reg)

In [ ]:
# Celula 7 — ALBUNS
print('--- Albuns')
S = {'Nome do Album':None,'Nome do Artista':None,'Capa do Album':None,'Link do audio':None,'telegram_topic_id':None,'data_lancamento':None}
rows = ler_aba('Albuns')
reg = []
for r in rows:
    nome = r.get('Nome')
    if not nome: continue
    reg.append(norm(S, {
        'Nome do Album': nome,
        'Nome do Artista': r.get('Nome do criador'),
        'Capa do Album': r.get('Capa'),
        'Link do audio': r.get('Link do audio'),
        'telegram_topic_id': n(r.get('ID do topico')),
        'data_lancamento': str(r.get('Data de lancamento', '') or '') or None,
    }))
upsert('Albuns', reg)

In [ ]:
# Celula 8 — MUSIC VIDEOS
print('--- Music Videos')
S = {'Titulo':None,'Artista':None,'Capa':None,'telegram_file_id':None,'telegram_topic_id':None,'tipo':None,'data_lancamento':None}
rows = ler_aba('Music Videos')
reg = []
for r in rows:
    titulo = r.get('Nome') or r.get('Titulo')
    if not titulo: continue
    reg.append(norm(S, {
        'Titulo': titulo,
        'Artista': r.get('Nome do criador'),
        'Capa': r.get('Thumb'),
        'telegram_file_id': r.get('ID do arquivo'),
        'telegram_topic_id': n(r.get('ID do topico')),
        'tipo': r.get('Tipo'),
        'data_lancamento': str(r.get('Data de lancamento', '') or '') or None,
    }))
upsert('Music Videos', reg)

In [ ]:
# Celula 9 — VIDEOS
print('--- Videos')
S = {'Titulo':None,'Artista':None,'Capa':None,'telegram_file_id':None,'telegram_topic_id':None}
rows = ler_aba('Videos')
reg = []
for r in rows:
    titulo = r.get('titulo') or r.get('Titulo') or r.get('Nome')
    if not titulo: continue
    reg.append(norm(S, {
        'Titulo': titulo,
        'Artista': r.get('artista') or r.get('Artista') or r.get('enviado_por'),
        'Capa': r.get('thumbnail_url') or r.get('Thumb') or r.get('Capa'),
        'telegram_file_id': r.get('ID do arquivo') or r.get('telegram_file_id'),
        'telegram_topic_id': n(r.get('ID do topico') or r.get('telegram_topic_id')),
    }))
upsert('Videos', reg)

In [ ]:
# Celula 10 — TOP 50 SPOTIFY
print('--- Top_50_Spotify')
S = {'posicao':None,'nome_musica':None,'capa_musica':None,'link_audio':None,'telegram_topic_id':None}
rows = ler_aba('Top_50_Spotify')
reg = []
for i,r in enumerate(rows):
    nome = r.get('Nome da musica') or r.get('Nome')
    if not nome: continue
    reg.append(norm(S, {
        'posicao': n(r.get('Posicao')) or i+1,
        'nome_musica': nome,
        'capa_musica': r.get('Capa da musica'),
        'link_audio': r.get('Link do audio'),
        'telegram_topic_id': n(r.get('ID do topico')),
    }))
upsert('Top_50_Spotify', reg)

In [ ]:
# Celula 11 — TOP APPLE MUSIC
print('--- Top_Songs_Apple_Music')
S = {'posicao':None,'nome_musica':None,'capa_musica':None,'link_audio':None,'telegram_topic_id':None}
rows = ler_aba('Top_Songs_Apple_Music')
reg = []
for i,r in enumerate(rows):
    nome = r.get('Nome da musica') or r.get('Nome')
    if not nome: continue
    reg.append(norm(S, {
        'posicao': n(r.get('Posicao')) or i+1,
        'nome_musica': nome,
        'capa_musica': r.get('Capa da musica'),
        'link_audio': r.get('Link do audio'),
        'telegram_topic_id': n(r.get('ID do topico')),
    }))
upsert('Top_Songs_Apple_Music', reg)

In [ ]:
# Celula 12 — TOP VIDEOS YT
print('--- Top_Videos_YT')
S = {'posicao':None,'nome_video':None,'thumb':None,'link_audio':None,'telegram_topic_id':None}
rows = ler_aba('Top_Videos_YT')
reg = []
for i,r in enumerate(rows):
    nome = r.get('Nome do video') or r.get('Nome')
    if not nome: continue
    reg.append(norm(S, {
        'posicao': n(r.get('Posicao')) or i+1,
        'nome_video': nome,
        'thumb': r.get('Thumb'),
        'link_audio': r.get('Link do audio'),
        'telegram_topic_id': n(r.get('ID do topico')),
    }))
upsert('Top_Videos_YT', reg)

In [ ]:
# Celula 13 — COMENTARIOS MUSICAS
print('--- Comentarios_Musicas')
S = {'telegram_topic_id':None,'id_jogador':None,'nome_jogador':None,'comentario':None}
rows = ler_aba('Comentarios_Musicas')
reg = []
for r in rows:
    c = r.get('Comentario') or r.get('comentario')
    if not c: continue
    reg.append(norm(S, {
        'telegram_topic_id': n(r.get('ID do topico')),
        'id_jogador': str(r['ID do jogador']) if r.get('ID do jogador') else None,
        'nome_jogador': r.get('Nome do jogador'),
        'comentario': c,
    }))
upsert('Comentarios_Musicas', reg)

In [ ]:
# Celula 14 — COMENTARIOS MV
print('--- Comentarios_MV')
S = {'telegram_topic_id':None,'id_jogador':None,'nome_jogador':None,'comentario':None,'data':None}
rows = ler_aba('Comentarios_MV')
reg = []
for r in rows:
    c = r.get('Comentario') or r.get('comentario')
    if not c: continue
    reg.append(norm(S, {
        'telegram_topic_id': n(r.get('ID do topico')),
        'id_jogador': str(r['ID do jogador']) if r.get('ID do jogador') else None,
        'nome_jogador': r.get('Nome do jogador'),
        'comentario': c,
        'data': str(r.get('Data', '') or '') or None,
    }))
upsert('Comentarios_MV', reg)

In [ ]:
# Celula 15 — COMENTARIOS VIDEOS
print('--- Comentarios_Videos')
S = {'telegram_topic_id':None,'telegram_message_id':None,'texto':None,'autor':None,'id_usuario':None,'data':None,'reacoes':None}
rows = ler_aba('Comentarios_Videos')
reg = []
for r in rows:
    t = r.get('texto') or r.get('Comentario')
    if not t: continue
    reg.append(norm(S, {
        'telegram_topic_id': n(r.get('telegram_topic_id')),
        'telegram_message_id': n(r.get('telegram_message_id')),
        'texto': t,
        'autor': r.get('autor'),
        'id_usuario': str(r['id_usuario']) if r.get('id_usuario') else None,
        'data': str(r.get('data', '') or '') or None,
        'reacoes': str(r.get('reacoes', '') or '') or None,
    }))
upsert('Comentarios_Videos', reg)

In [ ]:
# Celula 16 — COMENTARIOS ALBUNS
print('--- Comentarios_Albuns')
S = {'telegram_topic_id':None,'id_jogador':None,'nome_jogador':None,'comentario':None,'data':None}
rows = ler_aba('Comentarios_Albuns')
reg = []
for r in rows:
    c = r.get('Comentario') or r.get('comentario')
    if not c: continue
    reg.append(norm(S, {
        'telegram_topic_id': n(r.get('ID do topico')),
        'id_jogador': str(r['ID do jogador']) if r.get('ID do jogador') else None,
        'nome_jogador': r.get('Nome do jogador'),
        'comentario': c,
        'data': str(r.get('Data', '') or '') or None,
    }))
upsert('Comentarios_Albuns', reg)

print('\nIMPORTACAO CONCLUIDA!')